# AEVB MNIST Reproduction v3

This notebook runs a paper-aligned practical reproduction of Kingma and Welling's Auto-Encoding Variational Bayes experiments on MNIST. The implementation lives in `src/aevb_v3`; this notebook is intentionally a thin, reproducible Colab driver.

## Scope

- Main dataset: statically binarized MNIST with seed `2026`.
- Main method: AEVB/VAE with one hidden layer, `tanh`, Gaussian encoder, Bernoulli decoder, Adagrad, and explicit MAP L2 penalty.
- Baseline: Wake-Sleep with the same architecture and optimizer.
- Low-dimensional experiment: `z_dim=3`, `hidden_dim=100`, HMC marginal likelihood estimate and MCEM baseline.
- Reliability: metrics are flushed during training, intermediate checkpoints support resume, and the suite runner backs up results after each stage.
- Evaluation scope follows the original paper rather than adding modern diagnostics such as active units or IWAE estimates.

In [ ]:
from pathlib import Path
import os
import shutil

PROJECT_NAME_CANDIDATES = [
    'AEVB_MNIST_Reproduction_v3',
    'AEVB_MNIST_Reproduction_v3_Submission',
]
LOCAL_PROJECT_ROOT = Path('/content') / 'AEVB_MNIST_Reproduction_v3'
DRIVE_OUTPUT_DIR = None

try:
    from google.colab import drive
    IN_COLAB = True
except Exception:
    IN_COLAB = False

def is_project_root(path: Path) -> bool:
    return (
        (path / 'pyproject.toml').exists()
        and (path / 'scripts').exists()
        and (path / 'src' / 'aevb_v3').exists()
    )

def find_drive_project_root():
    drive_root = Path('/content/drive/MyDrive')
    candidates = []
    for project_name in PROJECT_NAME_CANDIDATES:
        candidates.extend([
            drive_root / 'ASI' / 'Assignment' / project_name,
            drive_root / project_name,
        ])
    for candidate in candidates:
        if is_project_root(candidate):
            return candidate

    matches = [p.parent for p in drive_root.rglob('pyproject.toml') if is_project_root(p.parent)]
    matches = [p for p in matches if 'aevb' in p.name.lower()] or matches
    if not matches:
        raise FileNotFoundError(
            'Could not find the AEVB v3 project in Google Drive. Upload the whole project folder, not only this notebook.'
        )
    return matches[0]

if IN_COLAB:
    drive.mount('/content/drive')
    DRIVE_PROJECT_ROOT = find_drive_project_root()
    DRIVE_OUTPUT_DIR = DRIVE_PROJECT_ROOT.parent
    LOCAL_PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
    for name in ['configs', 'scripts', 'src', 'notebooks']:
        src = DRIVE_PROJECT_ROOT / name
        dst = LOCAL_PROJECT_ROOT / name
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
    for name in ['README.md', 'requirements.txt', 'pyproject.toml', '.gitattributes', '.gitignore']:
        shutil.copy2(DRIVE_PROJECT_ROOT / name, LOCAL_PROJECT_ROOT / name)
    os.chdir(LOCAL_PROJECT_ROOT)
else:
    if not (Path.cwd() / 'pyproject.toml').exists():
        raise RuntimeError('Run this notebook from the repository root.')

PROJECT_ROOT = Path.cwd()
print('Project root:', PROJECT_ROOT)
print('Drive project root:', DRIVE_PROJECT_ROOT if IN_COLAB else PROJECT_ROOT)
print('Drive output directory:', DRIVE_OUTPUT_DIR)


In [ ]:
!pip install -q -r requirements.txt
!pip install -q -e .

In [ ]:
import torch
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## Recommended Suite Run

This is the recommended entrypoint. It runs smoke tests, the learning-rate pilot, AEVB, Wake-Sleep, marginal likelihood, figures, result collection, and a Drive backup after every stage.

In [ ]:
!python scripts/run_v3_suite.py --backup-dir /content/drive/MyDrive/AEVB_MNIST_Reproduction_v3_backups

## Emergency Backup

Run this cell whenever you want to archive the current `results/` folder manually.

In [ ]:
!python scripts/backup_results.py --results-dir results --output-dir /content/drive/MyDrive/AEVB_MNIST_Reproduction_v3_backups --name AEVB_MNIST_Reproduction_v3_emergency

## Experiment 1: AEVB Lower-Bound Sweep

This runs `z_dim=[3,5,10,20,200]` with three seeds plus `z_dim=2` with seed 0 for the latent manifold figure. The full configuration uses 300 epochs.

In [ ]:
# Optional manual command; skip this if you used the suite runner above.
# !python scripts/run_aevb_sweep.py --config configs/aevb_binarized_practical.yaml --resume

## Experiment 2: Wake-Sleep Baseline

Wake-Sleep is trained with the same MLP architecture, initialization, batch size, optimizer, and latent dimensions as AEVB, using seed 0 for the baseline comparison. The same ELBO evaluator is used for comparison.

In [ ]:
# Optional manual command; skip this if you used the suite runner above.
# !python scripts/run_wake_sleep.py --config configs/wake_sleep_binarized_practical.yaml --resume

## Experiment 3: Low-Dimensional Marginal Likelihood

This experiment follows the paper's low-dimensional setting with a focused configuration: `z_dim=3`, `hidden_dim=100`, `N_train=1000`, AEVB, Wake-Sleep, and MCEM. It estimates marginal likelihood with HMC posterior samples and a full-covariance Gaussian density estimator.

In [ ]:
# Optional manual command; skip this if you used the suite runner above.
# !python scripts/run_marginal_likelihood.py --config configs/marginal_likelihood_z3_h100_practical.yaml --resume

## Figures and Result Tables

In [ ]:
!python scripts/make_figures.py --config configs/aevb_binarized_practical.yaml
!python scripts/collect_results.py --results-dir results

In [ ]:
from pathlib import Path
for subdir in ['metrics', 'tables', 'figures', 'manifests']:
    files = sorted((Path('results') / subdir).glob('*'))
    print(f'{subdir}: {len(files)} files')
    for path in files[:10]:
        print(' -', path)

## Export Results

The result folder should be committed or archived with the code. Checkpoints can be stored with Git LFS or uploaded as a GitHub Release asset.

In [ ]:
!zip -r AEVB_MNIST_Reproduction_v3_results.zip results README.md configs scripts src pyproject.toml requirements.txt .gitattributes .gitignore

if DRIVE_OUTPUT_DIR is not None:
    destination = Path(DRIVE_OUTPUT_DIR) / 'AEVB_MNIST_Reproduction_v3_results.zip'
    shutil.copy2('AEVB_MNIST_Reproduction_v3_results.zip', destination)
    print('Copied archive to:', destination)